# TWYD End-to-End (Colab + Drive portable)
Train What You Deploy: Dense-LRC / CORE-LRC on ms-swift.
Repo: https://github.com/wenhui-ml/twyd (public)


In [ ]:
import os
ROOT = os.environ.get('TWYD_ROOT', '')
CANDS = ['/content/drive/MyDrive/Papers/twyd', '/content/twyd', os.getcwd()]
for c in CANDS:
    _t = os.path.join(c, 'twyd', '__init__.py')
    if not ROOT and os.path.isfile(_t):
        ROOT = c
if not ROOT:
    ROOT = '/content/drive/MyDrive/Papers/twyd'
print('ROOT =', ROOT)
import torch
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())


## 0a. mount Drive (Colab only; local: skip)
Mount once per session so ROOT persists on Drive.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted')
except Exception as e:
    print('skip mount (local run):', e)


## 0b. fetch code if ROOT empty (fixes empty-Drive error)
Fresh Colab Drive dir is empty -> auto git clone public repo. Rerun-safe.


In [ ]:
%%bash -s "$ROOT"
ROOT="$1"
REPO="https://github.com/wenhui-ml/twyd.git"
mkdir -p "$ROOT"
if [ ! -f "$ROOT/twyd/__init__.py" ]; then
  echo "cloning $REPO -> $ROOT"
  rm -rf "$ROOT.tmp" && git clone --depth 1 "$REPO" "$ROOT.tmp" 2>&1 | tail -n 3
  cp -r "$ROOT.tmp/." "$ROOT/" && rm -rf "$ROOT.tmp"
else
  echo "code present, skip clone"
fi
ls "$ROOT"


## 0c. env install (guarded: SKIP not fail)


In [ ]:
%%bash -s "$ROOT"
ROOT="$1"
if [ ! -f "$ROOT/twyd/__init__.py" ]; then echo "SKIP: run 0b first"; exit 0; fi
pip install -e "$ROOT" -q 2>&1 | tail -n 2
python -c "import twyd.plugin as P; print(sorted(P._TUNERS))"


## 1. checks (guarded)


In [ ]:
%%bash -s "$ROOT"
ROOT="$1"
if [ ! -d "$ROOT/tests" ]; then echo "SKIP: run 0b first"; exit 0; fi
python -m pytest "$ROOT/tests/" -q 2>&1 | tail -n 3


## 2. train ($ROOT; GPU for real run)
--tuner_type dense_lrc|core_lrc|plain_lrc --loss_type lrc_distill --external_plugins twyd.plugin


In [ ]:
%%bash -s "$ROOT"
ROOT="$1"
if [ ! -f "$ROOT/scripts/train_dense.sh" ]; then echo SKIP; exit 0; fi
cat "$ROOT/scripts/train_dense.sh"


## 3. convert (merge to HF; guarded)


In [ ]:
import os, sys
R = globals().get('ROOT', os.getcwd())
if R not in sys.path: sys.path.insert(0, R)
try:
    from twyd.merge import merged_state_dict
    print('merge import ok')
except ModuleNotFoundError as e:
    print('SKIP:', e)


## 4. eval (CKPT under $ROOT/outputs; GPU)
swift eval --eval_backend Native --eval_dataset gsm8k arc --eval_limit 10


In [ ]:
%%bash -s "$ROOT"
ROOT="$1"
echo "ROOT=$ROOT"
echo "uncomment on GPU: swift eval --model $ROOT/outputs/dense-05b-deploy --eval_backend Native --eval_dataset gsm8k arc --eval_limit 10"


## 5. fix table
| symptom | fix |
|---|---|
| Drive empty | run 0b auto git clone https://github.com/wenhui-ml/twyd |
| KL=None | wrapper _twyd_last_inputs fallback |
| Qwen collapse | outlier_keep=48 |
